In [9]:
import numpy as np
import random
import xgboost as xgb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [10]:
df_improvement = pd.read_csv("dataset/df_clean.csv")

# Although XGBoost doesn't need scaled inputs, put it anyways to ensure fairness for comparison between baseline and improved model
df_improvement["Minutes of Use"] = df_improvement["Seconds of Use"] / 60
df_improvement["Charge Amount/10"] = df_improvement["Charge  Amount"] / 10
df_improvement = df_improvement.drop(columns=["Seconds of Use", "Charge  Amount"])

print(f"Shape: {df_improvement.shape}")
df_train, df_test = train_test_split(df_improvement, test_size=0.4, stratify=df_improvement['Status'], random_state=42)
print(f"Shape for training set: {df_train.shape}")
print(f"Shape for test set: {df_test.shape}")

Shape: (2850, 14)
Shape for training set: (1710, 14)
Shape for test set: (1140, 14)


In [11]:
# Drop the targets for X so that it prevents data leakage 
X_train = df_train.drop(columns=["Subscription  Length", "Churn"])

# DMatrix just makes the XGB perform faster later during training
dtrain = xgb.DMatrix(X_train)

# Setting up the boundaries
# We know that no customers churned before the end of their Subscription Length (so it becomes the lower bound)
y_lower_bound = df_train['Subscription  Length'].values

# Where function is like the if() function, params are (condition, result if positive, result if negative)
# Let's say if the result is positive, the model knows the customer will churn eventually, 
# but in the meantime, assume they churn between month of their subs length and month infinity or forever
# if negative, it means they churned around the month of their subs length
y_upper_bound = np.where(df_train['Churn'] == 0, np.inf, df_train['Subscription  Length'].values)

dtrain.set_float_info('label_lower_bound', y_lower_bound)
dtrain.set_float_info('label_upper_bound', y_upper_bound)

X_test = df_test.drop(columns=["Subscription  Length", "Churn"])
dtest = xgb.DMatrix(X_test)

y_lower_test = df_test['Subscription  Length'].values
y_upper_test = np.where(df_test['Churn'] == 0, np.inf, df_test['Subscription  Length'].values)

dtest.set_float_info('label_lower_bound', y_lower_test)
dtest.set_float_info('label_upper_bound', y_upper_test)

In [ ]:
# Hyperparameter dictionary for survival analysis
parameters = {
    'objective': 'survival:aft',
    'eval_metric': 'aft-nloglik',
    'aft_loss_distribution': 'normal',
    'learning_rate': 0.05,
    'max_depth': 4,
    'min_child_weight': 50
}

# Write evals here just for neatness
evals = [(dtrain, 'train'), (dtest, 'test')]

xgb_model = xgb.train(
    params=parameters, 
    dtrain=dtrain, 
    num_boost_round=500, 
    evals=evals, 
    early_stopping_rounds=20,
    verbose_eval=10  
)

In [ ]:
# Implementing Random Search for finding the best hyperparameter set (Hyperparameter Tuning)
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'min_child_weight': [10, 30, 50, 70, 100]
}

# Set up on how many different combinations of hyperparameters to test
n_iter = 20
tune_res = []

for i in range(n_iter):
    lr = random.choice(param_grid['learning_rate'])
    md = random.choice(param_grid['max_depth'])
    mcw = random.choice(param_grid['min_child_weight'])

    parameters = {
        'objective': 'survival:aft',
        'eval_metric': 'aft-nloglik',
        'aft_loss_distribution': 'normal', # This is the best distribution based on the test earlier
        'learning_rate': lr,
        'max_depth': md,
        'min_child_weight': mcw
    } 

    model = xgb.train(
        params=parameters, 
        dtrain=dtrain, 
        num_boost_round=500, 
        evals=[(dtrain, 'train'), (dtest, 'test')], 
        early_stopping_rounds=20,
        verbose_eval=False
    )

    tune_res.append({
        'Iteration': i+1,
        'Learning Rate': lr, 
        'Max Depth': md, 
        'Min Child Weight': mcw, 
        'Error': model.best_score
    })

# Convert to DataFrame and sort all the values so that we can find the best combinations of parameters
results_df = pd.DataFrame(tune_res).sort_values(by='Error', ascending=True)
print("\n--- Top 3 Parameter Combinations ---")
print(results_df.head(3))


--- Top 3 Parameter Combinations ---
    Iteration  Learning Rate  Max Depth  Min Child Weight     Error
4           5            0.2          5                10  0.809891
7           8            0.1          5                10  0.811285
28         29            0.1          4                10  0.812337
